# NB_Purview_Copilot_BronzeToSilver
**Source:** `AI_Medallion.Bronze.purview_copilot_usage` (OneLake deltalake scan, same as Cursor bronze)  
**Targets (reads):** `okta_user_for_ai`, `hc_silver_historical`  
**Target:** `AI_Medallion.Silver.purview_copilot_usage` (+ `purview_copilot_usage_okta_reject`)  
**Run after:** Okta + HC silver notebooks  

**Run control:** `COPILOT_CLOSE_MONTH_NO` — month-close filters CreationDate through that month and replaces that month in silver; daily uses `PURVIEW_COPILOT_MIN_CREATION_DATE`. Frozen months: `Gold.ingest_freeze_control` (`product` copilot|all).  

**Local Windows:** `publish_merge_staging` **collapses** the heavy Purview join to JVM parquet first, then stages (avoids `toLocalIterator` on the full plan — #146). Okta/HC reads stay plain `read_table_broken_lineage`. One publish only; no extra `break_lineage_local` on the silver frame.

In [ ]:
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
REBUILD_MEDALLION_CACHE = False  # True: rescan OneLake; False: use ~/.fabric/medallion_cache (after first good bronze scan)
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("PYSPARK_DRIVER_MEMORY", "12g")
os.environ.setdefault("MEDALLION_COLLAPSE_PARTITIONS", "8")
os.environ.setdefault("MEDALLION_STAGING_SPARK_PARQUET", "1")
os.environ.setdefault("MEDALLION_STAGING_WRITE_PARTITIONS", "16")
os.environ.setdefault("MEDALLION_STAGING_COLLECT_ROWS", "15000")
os.environ["MEDALLION_STAGING_COLLECT_PARTITIONS"] = "8"
os.environ["MEDALLION_STAGING_CHECKPOINT"] = "0"  # localCheckpoint OOMs on large Purview joins (#146)
os.environ.setdefault("MEDALLION_STAGING_PARTITION_WISE", "1")
# Month-close YYYYMM: OneLake scan reads ONLY that CreationDate month (not full bronze history).
COPILOT_CLOSE_MONTH_NO = None
# COPILOT_CLOSE_MONTH_NO = 202609
from Common.io import sync_copilot_close_month_env, sync_rebuild_medallion_cache_env

sync_rebuild_medallion_cache_env(REBUILD_MEDALLION_CACHE)
sync_copilot_close_month_env(COPILOT_CLOSE_MONTH_NO)


In [ ]:
%run ./Common/bootstrap


In [ ]:
_CLOSE_BRONZE = copilot_close_month_no()
_bronze_scan_predicate = copilot_bronze_scan_predicate(_CLOSE_BRONZE)
if _CLOSE_BRONZE is not None:
    y, m = divmod(int(_CLOSE_BRONZE), 100)
    print(
        f"purview bronze: month-close {_CLOSE_BRONZE} — OneLake scan limited to {y}-{m:02d} only",
        flush=True,
    )
elif _bronze_scan_predicate:
    print(
        f"purview bronze: daily scan from min CreationDate ({_bronze_scan_predicate})",
        flush=True,
    )
bronze = read_table_broken_lineage(
    TABLE_PURVIEW_COPILOT_BRONZE,
    "purview_copilot_bronze",
    scan_predicate=_bronze_scan_predicate,
    scan_columns=PURVIEW_COPILOT_BRONZE_SCAN_COLUMNS,
)


In [ ]:
okta = read_table_broken_lineage(TABLE_OKTA_SILVER, "okta_silver")
okta = filter_through_close_month_ts(
    okta, "OktaUserUpdatedAt", copilot_close_month_no()
)


In [ ]:
hc = read_table_broken_lineage(TABLE_HC_SILVER, "hc_silver")
hc = filter_to_month_no(hc, "Month_No", copilot_close_month_no())


In [ ]:
_CLOSE = copilot_close_month_no()
_FROZEN = copilot_frozen_month_nos_to_skip(spark, _CLOSE)
print(f"copilot silver: close_month={_CLOSE!r} frozen_skip={_FROZEN}", flush=True)

# Daily: [] = all bronze files. Month-close: optional filename filter.
COPILOT_BRONZE_FILENAMES = []

_creation_col = (
    PURVIEW_BRONZE_FILTER_DATE_COLUMN
    if PURVIEW_BRONZE_FILTER_DATE_COLUMN in bronze.columns
    else "creation_date"
)
if COPILOT_BRONZE_FILENAMES:
    _file_col = "FileName" if "FileName" in bronze.columns else "file_name"
    bronze = bronze.filter(F.col(_file_col).isin(COPILOT_BRONZE_FILENAMES))
bronze = filter_through_close_month_ts(bronze, _creation_col, _CLOSE)


In [ ]:
apply_local_merge_spark_conf(spark)

min_date = purview_copilot_min_creation_date(_CLOSE)
print(f"purview_copilot_usage: creation_date >= {min_date}", flush=True)

# Parse AuditData once → local parquet (slim rows), then join Okta/HC (avoids OOM on combined plan).
_parsed = parse_purview_copilot_bronze_base(
    bronze,
    min_creation_date=min_date,
    close_month_no=_CLOSE,
    frozen_month_nos=_FROZEN,
)
_parsed = collapse_lineage_to_local_parquet(spark, _parsed, "purview_bronze_parsed")
write_df, _copilot_okta_rejects = join_purview_copilot_to_silver(
    _parsed, okta, hc, close_month_no=_CLOSE
)
_copilot_batch_months = _distinct_month_nos(
    write_df, "purview_copilot_usage_month_no"
)
publish_transform_staging(write_df, TABLE_PURVIEW_COPILOT_SILVER)
if _copilot_okta_rejects.head(1):
    publish_transform_staging(
        _copilot_okta_rejects, TABLE_PURVIEW_COPILOT_OKTA_REJECT
    )


In [ ]:
_silver_staging = transform_staging_path(TABLE_PURVIEW_COPILOT_SILVER)
write_df = spark.read.parquet(str(_silver_staging))
_reject_staging = transform_staging_path(TABLE_PURVIEW_COPILOT_OKTA_REJECT)
if _reject_staging.is_dir() and any(_reject_staging.glob("*.parquet")):
    _rej = spark.read.parquet(str(_reject_staging))
else:
    _rej = spark.createDataFrame(
        [],
        "purview_copilot_usage_record_id string, purview_copilot_usage_month_no int, okta_reject_reason string, okta_reject_loaded_at timestamp",
    )
write_copilot_okta_rejects(spark, _rej, _copilot_batch_months)

if copilot_close_month_no() is not None:
    write_df = replace_one_month_in_frame(
        spark,
        TABLE_PURVIEW_COPILOT_SILVER,
        write_df,
        "purview_copilot_usage_month_no",
        int(copilot_close_month_no()),
    )
    write_full_table(write_df, TABLE_PURVIEW_COPILOT_SILVER)
else:
    write_copilot_silver_daily(
        spark,
        write_df,
        TABLE_PURVIEW_COPILOT_SILVER,
        PURVIEW_COPILOT_MERGE_KEY,
        close_month_no=copilot_close_month_no(),
    )
